In [ ]:
# --- repo-root guard ---
# Notebooks live in notebooks/, but all data/code paths are relative to the repo root.
# If launched with the working directory set to notebooks/, step up one level so that
# 'src/', 'data/', 'pride_data/' and 'figures/' resolve correctly. Idempotent / no-op at root.
import os, pathlib
_cwd = pathlib.Path.cwd()
if not (_cwd / 'src').exists() and (_cwd.parent / 'src').exists():
    os.chdir(_cwd.parent)
print('working directory:', pathlib.Path.cwd())


In [40]:
# --- output directories (created relative to repo root) ---
from pathlib import Path as _P
for _d in ('figures','figures/figure2','figures/figure3','figures/figure4','figures/figure5'):
    _P(_d).mkdir(parents=True, exist_ok=True)

import sys, os
sys.path.insert(0, os.path.abspath("src"))
import re
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import importlib, core
importlib.reload(core)
from core import (count_sites_per_sample_ptm_report, process_ptm_site_report, _hex_to_rgba)

In [41]:
# Palettes  (FF = red, FFPE = blue), shared across Figure 4
FF_COLOR   = '#e64126'
FFPE_COLOR = '#846db1'
INPUTS = [10, 20, 50, 100, 200, 500, 1000]   # ng protein input (dilution series)
PROC_META = {'Protein_group', 'Gene_group', 'PTM_0_aa', 'PTM_pos', 'PTM_mult123',
             'PTM_flank', 'PTM_Collapse_key', 'PTM_localization', 'UPD_seq'}

# Data upload

In [42]:
# Revision Figure 4 data = wide PTM Site Reports (per-run localization at 0.75),
# fresh-frozen (FF) and FFPE mouse-brain dilution series, 10 ng - 1 ug, n=3 each.
RAW_DIR = Path('pride_data/analysis_data/revision/figure4')

def find_report(material, ng):
    # FF 1 ug is the repeat acquisition; all others follow the standard naming
    if material == 'FF' and ng == 1000:
        return next(RAW_DIR.glob('*dilser_FF_1000ng_repeat_Report.tsv'))
    return next(RAW_DIR.glob(f'*dilser_{material}_{ng}ng_Report.tsv'))

ff   = {ng: pd.read_csv(find_report('FF',   ng), sep='\t', low_memory=False) for ng in INPUTS}
ffpe = {ng: pd.read_csv(find_report('FFPE', ng), sep='\t', low_memory=False) for ng in INPUTS}
print('FF inputs  :', list(ff))
print('FFPE inputs:', list(ffpe))

FF inputs  : [10, 20, 50, 100, 200, 500, 1000]
FFPE inputs: [10, 20, 50, 100, 200, 500, 1000]


# Figure 4a
Class I phosphosite depth across the protein-input dilution series for fresh-frozen (FF) and FFPE mouse brain (n=3 per input). Strict per-run Class I, multiplicity collapsed, localization enforced.

In [43]:
# Figure 4a - Class I phosphosite depth vs input, FF and FFPE (grouped box + points, n=3).
ff_counts   = {ng: list(count_sites_per_sample_ptm_report(ff[ng]).values())   for ng in INPUTS}
ffpe_counts = {ng: list(count_sites_per_sample_ptm_report(ffpe[ng]).values()) for ng in INPUTS}

summary = pd.DataFrame({
    'FF_mean':   [int(np.mean(ff_counts[ng]))   for ng in INPUTS],
    'FFPE_mean': [int(np.mean(ffpe_counts[ng])) for ng in INPUTS],
}, index=INPUTS)
summary['FF/FFPE'] = (summary['FF_mean'] / summary['FFPE_mean']).round(2)
summary.index.name = 'input_ng'
print(summary.to_string())

xcat = [str(n) for n in INPUTS]
fig = go.Figure()
for label, counts, color in [('Fresh-frozen', ff_counts, FF_COLOR), ('FFPE', ffpe_counts, FFPE_COLOR)]:
    fig.add_trace(go.Box(
        y=[v for ng in INPUTS for v in counts[ng]],
        x=[str(ng) for ng in INPUTS for _ in counts[ng]],
        name=label, boxpoints='all', jitter=0.3, pointpos=0,
        marker=dict(size=7, color=color, line=dict(width=0.5, color='black')),
        line=dict(color=color, width=1.5), fillcolor=_hex_to_rgba(color, 0.15),
    ))
fig.update_layout(width=1000, height=600, template='plotly_white', boxmode='group',
                  xaxis_title='Protein input (ng)', yaxis_title='Class I phosphosites', showlegend = False)
fig.update_xaxes(categoryorder='array', categoryarray=xcat)
fig.update_yaxes(rangemode='tozero')
fig.show()
fig.write_image(r'figures/figure4/figure4a.pdf', width=1000, height=600)

          FF_mean  FFPE_mean  FF/FFPE
input_ng                             
10            453        126     3.60
20            921        227     4.06
50           1646        466     3.53
100          3303       1078     3.06
200          5086       2229     2.28
500          7878       3507     2.25
1000         8489       7283     1.17


# Figure 4b
Coefficient of variation of Class I phosphosite intensities across workflow replicates (n=3 per condition), FF vs FFPE, per protein input. CV is a per-feature quantitative metric — multiplicity is KEPT (per project policy). CV computed in linear space, requiring all 3 replicates valid.

In [44]:
# Figure 4b - replicate CV of Class I intensities, FF vs FFPE, per input (grouped box).
# Linear-space CV across the 3 reps, requiring all 3 valid; multiplicity KEPT.
def cv_per_input(reports):
    out = {}
    for ng in INPUTS:
        sd = process_ptm_site_report(reports[ng], cutoff=0.75)['site_data']
        cols = [c for c in sd.columns if c not in PROC_META]
        lin = np.power(2.0, sd[cols])
        n_valid = lin.notna().sum(axis=1)
        cv = lin.std(axis=1) / lin.mean(axis=1)
        cv = cv.where(n_valid >= len(cols))           # require all replicates valid
        out[ng] = cv.dropna()
    return out

cv_ff, cv_ffpe = cv_per_input(ff), cv_per_input(ffpe)
print('median CV  FF  : %.3f' % np.nanmedian(np.concatenate([v.values for v in cv_ff.values()])))
print('median CV  FFPE: %.3f' % np.nanmedian(np.concatenate([v.values for v in cv_ffpe.values()])))
print(pd.DataFrame({'FF_medianCV':   [round(float(np.nanmedian(cv_ff[ng])), 3)   for ng in INPUTS],
                    'FFPE_medianCV': [round(float(np.nanmedian(cv_ffpe[ng])), 3) for ng in INPUTS]},
                   index=INPUTS).to_string())

fig = go.Figure()
for label, cvd, color in [('Fresh-frozen', cv_ff, FF_COLOR), ('FFPE', cv_ffpe, FFPE_COLOR)]:
    fig.add_trace(go.Box(
        y=[v for ng in INPUTS for v in cvd[ng].values],
        x=[str(ng) for ng in INPUTS for _ in cvd[ng].values],
        name=label, boxpoints='outliers',
        marker=dict(size=3, color=color, opacity=0.4),
        line=dict(color=color), fillcolor=_hex_to_rgba(color, 0.15),
    ))
fig.update_layout(width=700, height=600, template='plotly_white', boxmode='group',
                  xaxis_title='Protein input (ng)', yaxis_title='Coefficient of variation', showlegend = False)
fig.update_xaxes(categoryorder='array', categoryarray=[str(n) for n in INPUTS])
fig.update_yaxes(range = [0, 1.81])
fig.show()
fig.write_image(r'figures/figure4/figure4b.pdf', width=700, height=600)

Dropped 201 non-Phospho rows. Mods in input: ['Carbamidomethyl (C)', 'Oxidation (M)', 'Phospho (STY)']. Mods kept: ['Phospho (STY)']
Per-run mask (cutoff=0.75): 0 of 1,402 intensity cells masked (0.0%).
Deduplicated multi-protein rows: 666 → 662.
Final: 662 sites × 3 samples.
Dropped 354 non-Phospho rows. Mods in input: ['Carbamidomethyl (C)', 'Oxidation (M)', 'Phospho (STY)']. Mods kept: ['Phospho (STY)']
Per-run mask (cutoff=0.75): 0 of 2,865 intensity cells masked (0.0%).
Deduplicated multi-protein rows: 1,435 → 1,431.
Final: 1,431 sites × 3 samples.
Dropped 446 non-Phospho rows. Mods in input: ['Carbamidomethyl (C)', 'Oxidation (M)', 'Phospho (STY)']. Mods kept: ['Phospho (STY)']
Per-run mask (cutoff=0.75): 0 of 5,211 intensity cells masked (0.0%).
Deduplicated multi-protein rows: 2,634 → 2,611.
Final: 2,611 sites × 3 samples.
Dropped 1,005 non-Phospho rows. Mods in input: ['Carbamidomethyl (C)', 'Oxidation (M)', 'Phospho (STY)']. Mods kept: ['Phospho (STY)']
Per-run mask (cutoff=0

# Figure 4c
Correlation between fresh-frozen and FFPE phosphoproteomes at 1 µg: scatter of log2-transformed Class I phosphosite intensities (mean of n=3 replicates per material). Only sites with Class I quantification in all three replicates of **both** materials are shown (complete cases). Pearson (linear) and Spearman (rank) are both reported — Spearman confirms the agreement is not driven by a few high/low-abundance sites or by FFPE dynamic-range compression.

In [45]:
# Figure 4c - FF vs FFPE Class I site-intensity correlation at 1 ug (complete cases).
from scipy import stats

sd1 = process_ptm_site_report(
    pd.read_csv(next(RAW_DIR.glob('*1ug_FF_versus_FFPE_Report.tsv')), sep='\t', low_memory=False),
    cutoff=0.75)['site_data']
sample_cols = [c for c in sd1.columns if c not in PROC_META]
ffpe_cols = [c for c in sample_cols if 'FFPE' in c]
ff_cols   = [c for c in sample_cols if 'FFPE' not in c and 'FF' in c]
print(f'FF cols={len(ff_cols)}  FFPE cols={len(ffpe_cols)}')

cmp = sd1[ff_cols + ffpe_cols].dropna()                      # complete cases in both materials
cmp = pd.DataFrame({'FF_mean': cmp[ff_cols].mean(axis=1),
                    'FFPE_mean': cmp[ffpe_cols].mean(axis=1)})
n = len(cmp)
pear_r, pear_p = stats.pearsonr(cmp['FFPE_mean'], cmp['FF_mean'])
spear_r, spear_p = stats.spearmanr(cmp['FFPE_mean'], cmp['FF_mean'])
slope, intercept, *_ = stats.linregress(cmp['FFPE_mean'], cmp['FF_mean'])
print(f'n sites (complete) = {n}')
print(f'Pearson  r = {pear_r:.3f}  (p = {pear_p:.1e})')
print(f'Spearman r = {spear_r:.3f}  (p = {spear_p:.1e})')

x_line = np.array([cmp['FFPE_mean'].min(), cmp['FFPE_mean'].max()])
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=cmp['FFPE_mean'], y=cmp['FF_mean'], mode='markers', showlegend=False,
    marker=dict(size=6, color='lightgrey', opacity=0.6, line=dict(color='#0F0E0D', width=0.2))))
fig.add_trace(go.Scatter(
    x=x_line, y=slope * x_line + intercept, mode='lines', showlegend=False,
    line=dict(color='red', width=3, dash='dash')))
fig.add_annotation(x=0.03, y=0.97, xref='paper', yref='paper', showarrow=False, align='left',
                   text=f'Pearson r = {pear_r:.2f}<br>Spearman r = {spear_r:.2f}<br>n = {n:,}')
fig.update_layout(width=600, height=600, template='plotly_white',
                  xaxis_title='FFPE log2 intensity', yaxis_title='Fresh-frozen log2 intensity')
fig.show()
fig.write_image(r'figures/figure4/figure4c.pdf', width=600, height=600)

Dropped 3,990 non-Phospho rows. Mods in input: ['Carbamidomethyl (C)', 'Oxidation (M)', 'Phospho (STY)']. Mods kept: ['Phospho (STY)']
Per-run mask (cutoff=0.75): 0 of 48,043 intensity cells masked (0.0%).
Deduplicated multi-protein rows: 14,203 → 14,176.
Final: 14,176 sites × 6 samples.
FF cols=3  FFPE cols=3
n sites (complete) = 3617
Pearson  r = 0.855  (p = 0.0e+00)
Spearman r = 0.842  (p = 0.0e+00)


# Figure 4d
Joint PCA of fresh-frozen and FFPE phosphoproteomes across the **full dilution series** (all inputs 10 ng–1 µg, n=3), addressing the reviewer request to show all concentrations together rather than a single input. No cross-run normalization (wo_norm export), consistent with the other dilution-series PCAs (2f, Suppl 1e–k). Class I per-run → set_condition → ≥70% valid-value filter → down-shifted imputation → PCA. Colored by material (FF red / FFPE blue), shaded by protein input.

In [46]:
# Figure 4d - joint FF/FFPE PCA across all inputs (wo_norm). Pipeline mirrors 3e/2f.
from collections import defaultdict
import analytics_core_V04 as ac

sd_all = process_ptm_site_report(
    pd.read_csv(next(RAW_DIR.glob('*FF_versus_FFPE_all_wo_norm_Report.tsv')), sep='\t', low_memory=False),
    cutoff=0.75)['site_data']
sample_cols = [c for c in sd_all.columns if c not in PROC_META]

def parse_cmp(name):
    material = 'FFPE' if 'FFPE' in name else 'FF'
    ng = int(re.search(r'_(\d+)ng', name).group(1))
    return material, ng

cond_to_samples = defaultdict(list)
for s in sample_cols:
    mat, ng = parse_cmp(s)
    cond_to_samples[f'{mat}_{ng}'].append(s)

rename_map, dict_cond = {}, {}
for cond, samples in cond_to_samples.items():
    base = cond.replace('_', '')                       # underscore-free sample IDs for set_condition
    ids = []
    for i, s in enumerate(sorted(samples)):
        sid = f'{base}c{i+1:02d}'; rename_map[s] = sid; ids.append(sid)
    dict_cond[cond] = ids

AC_META_REQUIRED = {'UPD_seq', 'PTM_localization', 'Protein_group', 'Gene_group', 'PTM_Collapse_key'}
sd_for_ac = sd_all.rename(columns=rename_map).drop(
    columns=[c for c in PROC_META if c not in AC_META_REQUIRED and c in sd_all.columns])

grouped = ac.set_condition(sd_for_ac, dict_cond)
filt    = ac.filt_per_percentage(grouped, 0.7)
imp     = ac.imputation_normal_distribution(filt).reset_index()
pca_ff  = ac.run_pca(imp)
print('groups:', imp['group'].nunique(), '| n samples:', len(imp))
print('explained variance:', pca_ff[1])

Dropped 5,080 non-Phospho rows. Mods in input: ['Carbamidomethyl (C)', 'Oxidation (M)', 'Phospho (STY)']. Mods kept: ['Phospho (STY)']
Per-run mask (cutoff=0.75): 0 of 175,064 intensity cells masked (0.0%).
Deduplicated multi-protein rows: 21,986 → 21,955.
Final: 21,955 sites × 42 samples.
groups: 14 | n samples: 42
explained variance: {'x_title': 'PC1 (0.68)', 'y_title': 'PC2 (0.05)', 'group': 'group'}


In [47]:
# Figure 4d plot - colored by material (FF red / FFPE violet), shaded by protein input.
pca_df = pca_ff[0][0]
INPUT_ORDER = [10, 20, 50, 100, 200, 500, 1000]
FF_SHADES   = ['#FCD0C5', '#FBA08D', '#FA7A61', '#F95534', '#ED2E07', '#CB2706', '#9E1E05']
FFPE_SHADES = ['#E3D1F5', '#C79EEA', '#B178E2', '#9B52DA', '#7E2AC7', '#6D25AD', '#551D87']

color_map = {}
for grp in pca_df['group'].unique():
    mat, ng = grp.rsplit('_', 1)
    shades = FF_SHADES if mat == 'FF' else FFPE_SHADES
    color_map[grp] = shades[INPUT_ORDER.index(int(ng))]

fig = px.scatter(pca_df, x='x', y='y', color='group', color_discrete_map=color_map)
fig.update_traces(marker=dict(size=15, line=dict(width=1, color='black')))
fig.update_layout(width=600, height=600, template='plotly_white', showlegend=False)
fig.update_xaxes(title=pca_ff[1]['x_title'])
fig.update_yaxes(title=pca_ff[1]['y_title'])
fig.show()
fig.write_image(r'figures/figure4/figure4d.pdf', width=600, height=600)

# Figure 4e
Pathway-level concordance of fresh-frozen and FFPE phosphoproteomes. For every KEGG pathway, Class I phosphosites are grouped by their parent protein (mouse KEGG annotation), and FF vs FFPE agreement is computed across that pathway's sites using both **Pearson** (linear/magnitude) and **Spearman** (rank, robust to FFPE dynamic-range compression) correlation of mean log2 intensities at 1 µg (complete cases, n=3 each). **Only pathways with ≥10 quantified sites are shown** (smaller pathways give unstable correlations). All KEGG pathways are plotted in grey; nine canonical signaling and brain/neuronal pathways are highlighted in red. Points clustering in the top-right near the diagonal indicate that the same pathways are recovered with the same quantitative behaviour from both materials.

*Methods note:* correlation inputs = per-site mean log2 Class I intensity (FF vs FFPE, 1 µg, complete cases); site→pathway grouping via parent-protein KEGG annotation (`mainAnnot.mus_musculus.txt`); pathways with ≥10 sites retained; both Pearson and Spearman reported. Notch and Jak-STAT signaling were excluded for <10 quantified Class I sites; Long-term potentiation and Axon guidance (relevant to brain tissue) are shown instead.

In [48]:
# Figure 4e - FF/FFPE per-pathway correlation across ALL KEGG pathways (>=10 sites).
from scipy import stats

ANNOT_PATH = r'data/mouse_kegg_annotation.tsv'
MIN_SITES  = 10
# canonical signaling + brain/neuronal pathways (Notch & Jak-STAT dropped: <10 Class I sites;
# replaced by Long-term potentiation and Axon guidance, both relevant to mouse brain)
HIGHLIGHT  = ['Wnt signaling pathway', 'MAPK signaling pathway', 'mTOR signaling pathway',
              'Calcium signaling pathway', 'ErbB signaling pathway', 'Insulin signaling pathway',
              'Neurotrophin signaling pathway', 'Long-term potentiation', 'Axon guidance']

# 1 ug FF vs FFPE Class I site means (complete cases), keep Protein_group for mapping
sd1 = process_ptm_site_report(
    pd.read_csv(next(RAW_DIR.glob('*1ug_FF_versus_FFPE_Report.tsv')), sep='\t', low_memory=False),
    cutoff=0.75)['site_data']
scols = [c for c in sd1.columns if c not in PROC_META]
ffpe_c = [c for c in scols if 'FFPE' in c]
ff_c   = [c for c in scols if 'FFPE' not in c and 'FF' in c]
keep = sd1[ff_c + ffpe_c].notna().all(axis=1)
sites = pd.DataFrame({'Protein_group': sd1.loc[keep, 'Protein_group'],
                      'FF_mean':   sd1.loc[keep, ff_c].mean(axis=1),
                      'FFPE_mean': sd1.loc[keep, ffpe_c].mean(axis=1)})

# parent-protein -> KEGG pathway mapping (exploded, deduplicated)
annot = pd.read_csv(ANNOT_PATH, sep='\t', low_memory=False)[['UniProt', 'KEGG name']].dropna()
annot = (annot.assign(Protein_group=annot['UniProt'].str.split(';'),
                      pathway=annot['KEGG name'].str.split(';'))
              .explode('Protein_group').explode('pathway')[['Protein_group', 'pathway']]
              .drop_duplicates())
merged = sites.merge(annot, on='Protein_group', how='inner')

rows = []
for path, g in merged.groupby('pathway'):
    if len(g) < MIN_SITES:
        continue
    rows.append({'pathway': path, 'n': len(g),
                 'Pearson':  stats.pearsonr(g['FFPE_mean'], g['FF_mean'])[0],
                 'Spearman': stats.spearmanr(g['FFPE_mean'], g['FF_mean'])[0]})
res = pd.DataFrame(rows)
res['highlight'] = res['pathway'].isin(HIGHLIGHT)
print(f'KEGG pathways with >= {MIN_SITES} sites: {len(res)}')
missing = [p for p in HIGHLIGHT if p not in set(res['pathway'])]
if missing:
    print('highlighted NOT shown (<min sites or unmatched):', missing)
print(res[res.highlight][['pathway', 'n', 'Pearson', 'Spearman']]
      .sort_values('pathway').round(3).to_string(index=False))
print('median (all shown): Pearson %.3f  Spearman %.3f' % (res['Pearson'].median(), res['Spearman'].median()))

lo = min(res[['Pearson', 'Spearman']].min().min(), 0)
g0, g1 = res[~res.highlight], res[res.highlight]
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=g0['Spearman'], y=g0['Pearson'], mode='markers', name='other KEGG pathways',
    marker=dict(size=6, color='lightgrey', line=dict(width=0.3, color='grey')),
    text=g0['pathway'], hovertemplate='%{text}<br>Sp=%{x:.2f}  Pe=%{y:.2f}<extra></extra>'))
fig.add_trace(go.Scatter(
    x=g1['Spearman'], y=g1['Pearson'], mode='markers', name='signaling / neuronal pathways',
    marker=dict(size=15, color='#EB420C', line=dict(width=1, color='DarkSlateGray')),
    text=g1['pathway'], hovertemplate='%{text}<br>Sp=%{x:.2f}  Pe=%{y:.2f}<extra></extra>'))
fig.add_shape(type='line', x0=lo, y0=lo, x1=1, y1=1,
              line=dict(color='darkgrey', width=2, dash='dash'))
fig.update_layout(width=680, height=600, template='plotly_white',
                  xaxis_title='Spearman r (FF vs FFPE)', yaxis_title='Pearson r (FF vs FFPE)', showlegend = False)
fig.update_xaxes(range=[lo - 0.02, 1.02]); fig.update_yaxes(range=[lo - 0.02, 1.02])
fig.show()
fig.write_image(r'figures/figure4/figure4e.pdf', width=600, height=600)

Dropped 3,990 non-Phospho rows. Mods in input: ['Carbamidomethyl (C)', 'Oxidation (M)', 'Phospho (STY)']. Mods kept: ['Phospho (STY)']
Per-run mask (cutoff=0.75): 0 of 48,043 intensity cells masked (0.0%).
Deduplicated multi-protein rows: 14,203 → 14,176.
Final: 14,176 sites × 6 samples.
KEGG pathways with >= 10 sites: 114
                       pathway   n  Pearson  Spearman
                 Axon guidance  42    0.814     0.773
     Calcium signaling pathway 103    0.855     0.842
        ErbB signaling pathway  71    0.891     0.890
     Insulin signaling pathway  46    0.841     0.811
        Long-term potentiation  79    0.868     0.863
        MAPK signaling pathway 124    0.890     0.870
Neurotrophin signaling pathway  55    0.778     0.813
         Wnt signaling pathway  63    0.915     0.916
        mTOR signaling pathway  17    0.803     0.821
median (all shown): Pearson 0.880  Spearman 0.856


# Figure 4f
Kernel density distribution of Class I phosphosite log2 intensities at 1 µg, fresh-frozen vs FFPE. Each site's value is the mean across its three replicates (sites quantified in all three replicates of that material). Comparable distribution shapes indicate that, although FFPE yields fewer sites, the quantitative dynamic range of the recovered phosphoproteome is preserved.

In [49]:
# Figure 4f - kernel density of Class I site intensities, FF vs FFPE at 1 ug.
# NOTE: figure_factory imported as 'pff' so it does not clobber the FF-reports dict 'ff'.
import plotly.figure_factory as pff

sdf = process_ptm_site_report(
    pd.read_csv(next(RAW_DIR.glob('*1ug_FF_versus_FFPE_Report.tsv')), sep='\t', low_memory=False),
    cutoff=0.75)['site_data']
scols = [c for c in sdf.columns if c not in PROC_META]
ffpe_c = [c for c in scols if 'FFPE' in c]
ff_c   = [c for c in scols if 'FFPE' not in c and 'FF' in c]

# per-material mean intensity over sites quantified in all 3 replicates of that material
ff_vals   = sdf.loc[sdf[ff_c].notna().all(axis=1),   ff_c].mean(axis=1)
ffpe_vals = sdf.loc[sdf[ffpe_c].notna().all(axis=1), ffpe_c].mean(axis=1)
print(f'FF sites   = {len(ff_vals):,}  median log2 = {ff_vals.median():.2f}')
print(f'FFPE sites = {len(ffpe_vals):,}  median log2 = {ffpe_vals.median():.2f}')

fig = pff.create_distplot(
    [ffpe_vals.tolist(), ff_vals.tolist()],
    group_labels=['FFPE', 'Fresh-frozen'], colors=[FFPE_COLOR, FF_COLOR],
    show_hist=True, show_rug=False, bin_size=0.2)
fig.update_layout(width=600, height=600, template='plotly_white',
                  xaxis_title='log2 phosphosite intensity', yaxis_title='Density', showlegend = False)
fig.show()
fig.write_image(r'figures/figure4/figure4f.pdf', width=600, height=600)

Dropped 3,990 non-Phospho rows. Mods in input: ['Carbamidomethyl (C)', 'Oxidation (M)', 'Phospho (STY)']. Mods kept: ['Phospho (STY)']
Per-run mask (cutoff=0.75): 0 of 48,043 intensity cells masked (0.0%).
Deduplicated multi-protein rows: 14,203 → 14,176.
Final: 14,176 sites × 6 samples.
FF sites   = 5,268  median log2 = 9.01
FFPE sites = 4,975  median log2 = 9.17


# Figure 4h–l — lung adenocarcinoma microbulk
Faithful reproduction of the Figure 4 v00 analytical pipeline on the revision **Class I** lungAC report (4h–l was already Class I in v00, `cutoff=0.75`; only the input report changed). Phosphosites are normalized to **parent-protein abundance per condition** (proteome DVP report) before PCA / differential analysis. Helper functions below are ported verbatim from v00.

In [50]:
# Lung-AC helper functions (ported verbatim from Figure4_v00).
import analytics_core_V04 as ac

def assign_condition_setup(collapsed_data, condition_df, key_col='PTM_Collapse_key'):
    """Reshape site_data into samples-as-rows long format (group/sample/subject)."""
    df_copy = collapsed_data.copy()
    quant_cols = condition_df['Sample'].unique().tolist()
    missing = [s for s in quant_cols if s not in df_copy.columns]
    if missing:
        raise KeyError(f"{len(missing)} samples not found: {missing[:3]}")
    meta_cols = [c for c in df_copy.columns if c not in quant_cols]
    quant_df, meta_df = df_copy[quant_cols].T, df_copy[meta_cols].T
    s2c = dict(zip(condition_df['Sample'], condition_df['Condition']))
    quant_df.columns = meta_df.loc[key_col]
    quant_df['group'] = quant_df.index.map(s2c)
    quant_df['sample'] = quant_df['group'] + '_' + (quant_df.groupby('group').cumcount() + 1).astype(str)
    quant_df['subject'] = quant_df['sample']
    return quant_df

def filter_per_condition_completeness(df, group_col='group', threshold=0.7,
                                      metadata_cols=('group', 'sample', 'subject'),
                                      min_n_per_group=2, verbose=True):
    """Keep sites with >= threshold valid fraction in at least one condition."""
    metadata_cols = tuple(c for c in metadata_cols if c in df.columns)
    site_cols = [c for c in df.columns if c not in metadata_cols]
    valid_frac = df.groupby(group_col)[site_cols].apply(lambda g: g.notna().mean())
    n_per_group = df.groupby(group_col).size()
    eligible = n_per_group[n_per_group >= min_n_per_group].index
    keep_mask = (valid_frac.loc[eligible] >= threshold).any(axis=0)
    keep_sites = set(keep_mask[keep_mask].index)
    out_cols = [c for c in df.columns if c in metadata_cols or c in keep_sites]
    if verbose:
        print(f"per-condition filter (>={threshold:.0%} in any of {sorted(eligible)}): "
              f"{len(site_cols):,} -> {len(keep_sites):,} sites")
    return df[out_cols]

def normalize_phospho_median(phospho_df, protein_df, return_non_matched=False):
    """Subtract each phosphosite's parent-protein median (per condition) from its intensity."""
    common = set(phospho_df.index) & set(protein_df.index)
    if not common:
        raise ValueError("No common conditions between phospho and protein data!")
    ph = phospho_df.loc[phospho_df.index.isin(common)]
    pr = protein_df.loc[protein_df.index.isin(common)]
    def extract(p): return p.split('~')[0] if '~' in p else p.split('_')[0]
    p2prot = {s: extract(s) for s in ph.columns}
    medians = pr.groupby(pr.index).median()
    out = ph.copy(); ok = set()
    for cond in common:
        mask = ph.index == cond
        for site in ph.columns:
            match = [c for c in medians.columns if p2prot[site] in c.split(';')]
            if match:
                m = medians.loc[cond, match[0]]
                if not pd.isna(m):
                    out.loc[mask, site] = ph.loc[mask, site] - m
                    ok.add(site)
    if not return_non_matched:
        out = out[list(ok)]
    print(f"normalize_phospho_median: {len(ok):,}/{len(ph.columns):,} sites matched to a parent protein")
    return {'normalized_phospho': out, 'common_conditions': sorted(common)}

# Figure 4h
PCA of the FFPE lung adenocarcinoma phosphoproteome (tumor = AC vs healthy-appearing = H), after sample QC, per-condition valid-value filtering, down-shifted imputation, and parent-protein normalization. *Note (reviewer #70): describe the separation conservatively from the explained variance printed below.*

In [51]:
# Figure 4h (part 1) - lungAC preprocessing: QC -> legend mapping -> filter -> impute.
from scipy import stats

LEGEND_PATH = r'tables/Supplementary_Table_3.xlsx'

lungac = process_ptm_site_report(
    pd.read_csv(next(RAW_DIR.glob('*nanoPhos_lungAC_Report.tsv')), sep='\t', low_memory=False),
    cutoff=0.75)['site_data']

# sample QC: drop columns whose site count < median - 1.5*IQR (low-quality samples)
sample_all = [c for c in lungac.columns if c not in PROC_META]
meta_cols  = [c for c in lungac.columns if c in PROC_META]
counts = {c: int(lungac[c].notna().sum()) for c in sample_all}
qc_thr = np.nanmedian(list(counts.values())) - stats.iqr(list(counts.values())) * 1.5
keep_samples = [c for c in sample_all if counts[c] >= qc_thr]
print(f'sample QC: {len(sample_all)} -> {len(keep_samples)} (threshold {qc_thr:.0f} sites)')
lungac_f = lungac[keep_samples + meta_cols]

# legend: tumor->AC, else H; sample_ID = type_patient (cohort embedded in patient); map via 384-well position
legend = pd.read_excel(LEGEND_PATH)
legend['sample_type'] = ['AC' if t == 'tumor (adenocarcinoma)' else 'H' for t in legend['sample_type']]
legend['sample_ID'] = legend['sample_type'] + '_' + legend['patient'].astype(str)
scd = pd.DataFrame({'Full_name': keep_samples,
                    '384_well_position': [c.split('_')[-1] for c in keep_samples]}).merge(
                    legend, on='384_well_position')
condition_df = pd.DataFrame({'Sample': scd['Full_name'], 'Condition': scd['sample_ID']})
print(f'samples mapped to legend: {len(condition_df)} | conditions: {condition_df["Condition"].nunique()} '
      f'| AC/H: {condition_df["Condition"].str.split("_").str[0].value_counts().to_dict()}')

# reshape, per-condition completeness filter (>=70% in AC or H), down-shifted imputation
df1 = assign_condition_setup(lungac_f, condition_df)
df1['group_reduced'] = df1['group'].apply(lambda x: x.split('_')[0])
df1f = filter_per_condition_completeness(df1, threshold=0.7, group_col='group_reduced').drop('group_reduced', axis=1)
imp = ac.imputation_normal_distribution(df1f).reset_index()
print(f'after imputation: {imp.shape[0]} samples x {imp.shape[1]-3} sites')

Dropped 1,600 non-Phospho rows. Mods in input: ['Carbamidomethyl (C)', 'Oxidation (M)', 'Phospho (STY)']. Mods kept: ['Phospho (STY)']
Per-run mask (cutoff=0.75): 0 of 200,291 intensity cells masked (0.0%).
Deduplicated multi-protein rows: 19,345 → 19,150.
Final: 19,150 sites × 62 samples.
sample QC: 62 -> 56 (threshold 1011 sites)
samples mapped to legend: 55 | conditions: 20 | AC/H: {'AC': 28, 'H': 27}
per-condition filter (>=70% in any of ['AC', 'H']): 19,151 -> 1,827 sites
after imputation: 55 samples x 1826 sites


In [52]:
# Figure 4h (part 2) - proteome load + parent-protein normalization.
# Proteome (DVP) report reused from the original analysis (protein-level, no Class I needed).
PROTEOME_PATH = r'pride_data/analysis_data/figure4/DVP_lungAC_Report.tsv'

prot = pd.read_csv(PROTEOME_PATH, sep='\t')
# fix the one malformed column (timestamp suffix) so its 384-well token parses
prot = prot.rename(columns={c: re.sub(r'_K13_\d+\.raw', '_K13.raw', c) for c in prot.columns})

# robust well -> sample_ID mapping (reproduces v00's manual drop of the 3 unmapped blank wells)
well2id = dict(zip(legend['384_well_position'].astype(str), legend['sample_ID']))
def _well(c): return c.split('_')[-1].split('.')[0]
phospho_ids = set(condition_df['Condition'])
prot_cols = [c for c in prot.columns if c != 'PG.ProteinGroups']
keep_prot = [c for c in prot_cols if well2id.get(_well(c)) in phospho_ids]
print(f'proteome cols: {len(prot_cols)} -> mapped & kept {len(keep_prot)} (blanks/unmatched dropped)')
prot_keep = prot[['PG.ProteinGroups'] + keep_prot].copy()
prot_keep.columns = ['PG.ProteinGroups'] + [well2id[_well(c)] for c in keep_prot]
prot_log = np.log2(prot_keep.set_index('PG.ProteinGroups').T)   # rows = condition samples, cols = proteins

# normalize each phosphosite by its parent-protein median within each condition
collapsed2 = imp.drop(['sample', 'subject'], axis=1).set_index('group')
norm = normalize_phospho_median(collapsed2, prot_log)['normalized_phospho'].reset_index()
norm['sample']  = imp['sample'].values
norm['subject'] = imp['subject'].values
norm = norm[norm['group'] != 'could_be_blank']
print(f'normalized phospho: {norm.shape[0]} samples x {norm.shape[1]-3} sites')

proteome cols: 63 -> mapped & kept 60 (blanks/unmatched dropped)
normalize_phospho_median: 1,747/1,826 sites matched to a parent protein
normalized phospho: 55 samples x 1747 sites


In [53]:
# Figure 4h (part 3) - PCA of normalized phosphoproteome, AC vs H, with 95% data ellipses + PERMANOVA.
from scipy.spatial.distance import pdist, squareform

norm1 = norm.copy()
norm1['group']   = norm1['group'].apply(lambda x: x.split('_')[0])                       # AC / H
norm1['sample']  = norm1['sample'].apply(lambda x: x.split('_')[0] + '_' + x.split('_')[-1])
norm1['subject'] = norm1['subject'].apply(lambda x: x.split('_')[0] + '_' + x.split('_')[-1])

pca_lung = ac.run_pca(norm1)
print('explained variance:', pca_lung[1])
_pdf = pca_lung[0][0]

# ---- PERMANOVA: AC vs H separation on the normalized feature matrix (Euclidean, 999 perm) ----
feat_cols = [c for c in norm1.columns if c not in ('group', 'sample', 'subject')]
X = norm1[feat_cols].values
labels = norm1['group'].values
D2 = squareform(pdist(X, 'euclidean')) ** 2

def _ssw(lab):
    s = 0.0
    for g in np.unique(lab):
        idx = np.where(lab == g)[0]; ng = len(idx)
        s += D2[np.ix_(idx, idx)][np.triu_indices(ng, 1)].sum() / ng
    return s

N = len(labels); a = len(np.unique(labels))
SST = D2[np.triu_indices(N, 1)].sum() / N
SSW = _ssw(labels); SSA = SST - SSW
F_obs = (SSA / (a - 1)) / (SSW / (N - a))
rng = np.random.default_rng(0)
count = 0
for _ in range(999):
    w = _ssw(rng.permutation(labels))
    if (SST - w) / (a - 1) / (w / (N - a)) >= F_obs:
        count += 1
permanova_p = (count + 1) / 1000
permanova_R2 = SSA / SST
print(f'PERMANOVA (AC vs H, Euclidean, 999 perm): pseudo-F={F_obs:.2f}  R2={permanova_R2:.3f}  p={permanova_p:.4f}')

# ---- plot with 95% data (dispersion) ellipses ----
CMAP = {'AC': '#C4291A', 'H': '#6020B3'}
def _ellipse(xs, ys, chi2_95=5.991, npts=120):
    cov = np.cov(xs, ys); mean = np.array([np.mean(xs), np.mean(ys)])
    vals, vecs = np.linalg.eigh(cov)
    t = np.linspace(0, 2 * np.pi, npts); circ = np.array([np.cos(t), np.sin(t)])
    ell = mean[:, None] + vecs @ (np.sqrt(chi2_95) * np.sqrt(vals)[:, None] * circ)
    return ell[0], ell[1]

fig = px.scatter(_pdf, x='x', y='y', color='group', color_discrete_map=CMAP)
fig.update_traces(marker=dict(size=21, line=dict(width=0.2, color='black')))
for g, c in CMAP.items():
    sub = _pdf[_pdf['group'] == g]
    ex, ey = _ellipse(sub['x'].values, sub['y'].values)
    fig.add_trace(go.Scatter(x=ex, y=ey, mode='lines', line=dict(color=c, width=2),
                             fill='toself', fillcolor=_hex_to_rgba(c, 0.10),
                             showlegend=False, hoverinfo='skip'))
fig.add_annotation(x=0.02, y=0.98, xref='paper', yref='paper', showarrow=False, align='left',
                   text=f'PERMANOVA p = {permanova_p:.3f}<br>R2 = {permanova_R2:.2f}')
fig.update_layout(width=600, height=600, template='plotly_white', showlegend=False)
fig.update_xaxes(title=pca_lung[1]['x_title'])
fig.update_yaxes(title=pca_lung[1]['y_title'])
fig.show()
fig.write_image(r'figures/figure4/figure4h.pdf', width=600, height=600)

explained variance: {'x_title': 'PC1 (0.18)', 'y_title': 'PC2 (0.08)', 'group': 'group'}
PERMANOVA (AC vs H, Euclidean, 999 perm): pseudo-F=6.88  R2=0.115  p=0.0010


# Figure 4i
Differential phosphorylation, tumor (AC) vs healthy (H), by `limma` on the parent-protein-normalized data, **blocked by patient** (paired design). Significance: |log2FC| > 0.585 (1.5×) and BH-adjusted p < 0.05. Volcano plot with up (red) / down (blue) / non-significant (grey).

In [54]:
# Figure 4i - limma AC vs H (patient-blocked) + volcano. (ported from v00)
from limma_utils import run_limma_pipeline

def mark_significant_sites(limma_output, log2_threshold=0.585, fdr_threshold=0.05):
    """5-way label: FDR-significant sites are 'up'/'down' if past the log2FC threshold,
    else 'up_sub'/'down_sub' (significant by FDR but sub-threshold fold change)."""
    df = limma_output.copy()
    tmp = []
    for _, row in df.iterrows():
        if row['adj.P.Val'] >= fdr_threshold:
            tmp.append('none')
        elif row['logFC'] > log2_threshold:
            tmp.append('up')
        elif row['logFC'] < -log2_threshold:
            tmp.append('down')
        elif row['logFC'] > 0:
            tmp.append('up_sub')
        else:
            tmp.append('down_sub')
    df['ID'] = tmp
    return df

# metadata + expression matrix (features x samples), exactly as v00
df_meta = norm[['group', 'sample', 'subject']].copy()
df_num  = norm.set_index('sample').drop(['group', 'subject'], axis=1).T
df_meta['sample_type'] = df_meta['group'].apply(lambda x: x.split('_')[0])
df_meta['patient_id']  = df_meta['group'].apply(lambda x: x.split('_')[1])   # e.g. 'c16/16012' (cohort embedded); blocks AC vs H per patient
df_meta.columns = ['group', 'sample_id', 'subject', 'sample_type', 'patient_id']
df_meta = df_meta.drop(['group', 'subject'], axis=1)

limma_res_AC_vs_H = run_limma_pipeline(
    df_num, df_meta, group_cols=['sample_type'], block_col='patient_id', lfc_cutoff=0.585)
limma_AC_vs_H = mark_significant_sites(limma_res_AC_vs_H)
limma_AC_vs_H['Gene'] = limma_AC_vs_H['feature'].apply(lambda x: x.split('~')[1].split('_')[0])

vc = limma_AC_vs_H['ID'].value_counts().to_dict()
n_up, n_down = vc.get('up', 0), vc.get('down', 0)
print(f'AC vs H (patient-blocked):  up={n_up}  down={n_down}  total sig (|log2FC|>0.585)={n_up + n_down}')
print(f'  FDR-sig but sub-threshold: up_sub={vc.get("up_sub",0)}  down_sub={vc.get("down_sub",0)}  '
      f'| of {len(limma_AC_vs_H)} tested')

COLOR_MAP = {'up': '#e40b0b', 'up_sub': '#fd9e02', 'down': '#126782', 'down_sub': '#e2eafc', 'none': '#d5d5d5'}
fig = px.scatter(limma_AC_vs_H, x='logFC', y=-np.log10(limma_AC_vs_H['adj.P.Val']), color='ID',
                 color_discrete_map=COLOR_MAP, hover_name='Gene',
                 category_orders={'ID': ['none', 'up_sub', 'down_sub', 'up', 'down']})
fig.update_traces(marker=dict(size=12, line=dict(width=0.2)))
fig.add_vline(x=0.585, line_dash='dash'); fig.add_vline(x=-0.585, line_dash='dash')
fig.add_hline(y=1.3, line_dash='dash')
fig.update_layout(width=600, height=600, template='plotly_white', showlegend=False)
fig.update_xaxes(title='log2FC (AC vs H)'); fig.update_yaxes(title='-log10(adj.P.Val)')
fig.show()
fig.write_image(r'figures/figure4/figure4i.pdf', width=600, height=600)

✓ ======================================================================
✓ COMPLETE LIMMA PIPELINE
✓ ======================================================================
📝 Logging to: limma_logs\limma_analysis_20260728_153822.log
✓ LIMMA ANALYSIS STARTED
✓ Design: 2 groups, 10 blocks. 10 blocks.
✓ Expression data filtered and aligned to 55 samples
✓ Sample order validated
✓ Converted to R: 1747 features × 55 samples


--- Logging error ---
Traceback (most recent call last):
  File "C:\Users\oliinyk\AppData\Local\Programs\Python\Python311\Lib\logging\__init__.py", line 1113, in emit
    stream.write(msg + self.terminator)
  File "C:\Users\oliinyk\AppData\Local\Programs\Python\Python311\Lib\encodings\cp1252.py", line 19, in encode
    return codecs.charmap_encode(input,self.errors,encoding_table)[0]
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeEncodeError: 'charmap' codec can't encode character '\u2713' in position 33: character maps to <undefined>
Call stack:
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "d:\Projects\nanoPhos_env\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "d:\Projects\nanoPhos_env\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "d:\Projects\nanoPhos_env\Lib\site-packages\ipykern

✓ Correlation: Correlation = 0.258.
✓ Effective sample size: Efficiency = 78.9%.
✓ Model fitted. eBayes: trend=True, robust=True
✓ LIMMA ANALYSIS COMPLETED
✓ Contrasts: 1 created
✓ Testing 1 regular contrasts + 0 trajectory F-tests (global FDR: True)...
✓ Results: 783 significant features (global FDR, adj.P < 0.05, |logFC| > 0.585)
✓ Results: 1747 rows (1747 features × 1 contrasts)
✓ Significant: 783 results (783 unique features)
✓ ======================================================================
✓ PIPELINE COMPLETED - Log: limma_logs\limma_analysis_20260728_153822.log
✓ ======================================================================
AC vs H (patient-blocked):  up=552  down=231  total sig (|log2FC|>0.585)=783
  FDR-sig but sub-threshold: up_sub=59  down_sub=14  | of 1747 tested


# Figure 4j
Kinase activity enrichment on the AC-vs-H differential phosphosites (Kinase Library, ser/thr, percentile-rank, threshold 15), using ±7-residue sequence windows from the human FASTA. Volcano of log2 frequency factor vs −log10 adjusted p; up-regulated (red) / down-regulated (blue) kinases.

In [55]:
# Figure 4j - Kinase Library enrichment on AC vs H differential. (ported verbatim from v00)
import kinase_library as kl
HUMAN_FASTA = r'pride_data/analysis_data/figure2/human.fasta'

def parse_fasta_with_gene_mapping(fasta_path):
    sequences, gene_to_accession = {}, {}
    acc = gene = None; seq = []
    with open(fasta_path) as f:
        for line in f:
            line = line.strip()
            if line.startswith('>'):
                if acc and seq:
                    sequences[acc] = ''.join(seq)
                    if gene: gene_to_accession[gene] = acc
                am = re.search(r'>(?:sp|tr)\|([A-Z0-9]+)\|', line)
                gm = re.search(r'GN=([A-Za-z0-9_-]+)', line)
                acc = am.group(1) if am else None
                gene = gm.group(1) if gm else None
                seq = []
            else:
                seq.append(line)
        if acc and seq:
            sequences[acc] = ''.join(seq)
            if gene: gene_to_accession[gene] = acc
    return sequences, gene_to_accession

def get_sequence_window(sequences, gene_to_accession, gene, position, window=7):
    if gene not in gene_to_accession: return None
    acc = gene_to_accession[gene]
    if acc not in sequences: return None
    seq = sequences[acc]; i = position - 1
    if i < 0 or i >= len(seq): return None
    start, end = i - window, i + window + 1
    lp = '_' * abs(min(0, start)); rp = '_' * max(0, end - len(seq))
    return lp + seq[max(0, start):min(len(seq), end)] + rp

sequences, gene_to_accession = parse_fasta_with_gene_mapping(HUMAN_FASTA)
limma_AC_vs_H['kinase_window'] = [
    get_sequence_window(sequences, gene_to_accession, row['Gene'], int(row['feature'].split('_')[1][1:]), window=7)
    for _, row in limma_AC_vs_H.iterrows()]

kin_in = limma_AC_vs_H[['feature', 'kinase_window', 'logFC', 'adj.P.Val']].copy()
kin_in.columns = ['Phosphosites', 'Sequence', 'logFC', 'adj.P.Val']
test = kl.DiffPhosData(kin_in, lfc_col='logFC', seq_col='Sequence', pval_col='adj.P.Val', pval_thresh=0.05)
kinase_df = test.kinase_enrichment(kin_type='ser_thr', kl_method='percentile_rank', kl_thresh=15).combined_enrichment_results

def _kid(r):
    sig = -np.log10(r['most_sig_fisher_adj_pval']) >= 1.3
    if sig and r['most_sig_log2_freq_factor'] >= 0: return 'upreg'
    if sig and r['most_sig_log2_freq_factor'] <= 0: return 'downreg'
    return 'noreg'
kinase_df['ID'] = kinase_df.apply(_kid, axis=1)
print('kinases enriched:', kinase_df['ID'].value_counts().to_dict())

fig = px.scatter(kinase_df, y=-np.log10(kinase_df['most_sig_fisher_adj_pval']),
                 x='most_sig_log2_freq_factor', color='ID',
                 color_discrete_map={'noreg': '#BDBDBD', 'upreg': '#EE4811', 'downreg': '#0B6299'},
                 hover_name=kinase_df.index)
fig.update_traces(marker=dict(size=10, line=dict(width=0.5, color='black')))
fig.add_hline(y=1.3, line_dash='dash', line_color='black')
fig.update_xaxes(title='log2 frequency factor (AC vs H)')
fig.update_yaxes(title='-log10(adjusted p)')
fig.update_layout(width=800, height=400, template='plotly_white', showlegend=False)
fig.show()
fig.write_image(r'figures/figure4/figure4j.pdf', width=800, height=400)

1 entries were omitted due to empty value in the substrates column.
1 entries were omitted due to invalid central phosphoacceptor.
Use the 'omited_entries' attribute to view dropped enteries due to invalid sequences.

Calculating percentiles for upregulated sites (611 substrates)
Scoring 601 ser_thr substrates
Calculating percentile for 601 ser_thr substrates
 20%|██        | 63/311 [00:00<00:00, 258.20it/s]

--- Logging error ---
Traceback (most recent call last):
  File "C:\Users\oliinyk\AppData\Local\Programs\Python\Python311\Lib\logging\__init__.py", line 1113, in emit
    stream.write(msg + self.terminator)
  File "C:\Users\oliinyk\AppData\Local\Programs\Python\Python311\Lib\encodings\cp1252.py", line 19, in encode
    return codecs.charmap_encode(input,self.errors,encoding_table)[0]
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeEncodeError: 'charmap' codec can't encode character '\u2588' in position 37: character maps to <undefined>
Call stack:
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "d:\Projects\nanoPhos_env\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "d:\Projects\nanoPhos_env\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "d:\Projects\nanoPhos_env\Lib\site-packages\ipykern

 37%|███▋      | 115/311 [00:00<00:00, 253.05it/s]

--- Logging error ---
Traceback (most recent call last):
  File "C:\Users\oliinyk\AppData\Local\Programs\Python\Python311\Lib\logging\__init__.py", line 1113, in emit
    stream.write(msg + self.terminator)
  File "C:\Users\oliinyk\AppData\Local\Programs\Python\Python311\Lib\encodings\cp1252.py", line 19, in encode
    return codecs.charmap_encode(input,self.errors,encoding_table)[0]
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeEncodeError: 'charmap' codec can't encode characters in position 37-39: character maps to <undefined>
Call stack:
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "d:\Projects\nanoPhos_env\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "d:\Projects\nanoPhos_env\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "d:\Projects\nanoPhos_env\Lib\site-packages\ipykernel\ke

 56%|█████▌    | 174/311 [00:00<00:00, 263.81it/s]

...

 84%|████████▍ | 261/311 [00:00<00:00, 273.75it/s]

--- Logging error ---
Traceback (most recent call last):
  File "C:\Users\oliinyk\AppData\Local\Programs\Python\Python311\Lib\logging\__init__.py", line 1113, in emit
    stream.write(msg + self.terminator)
  File "C:\Users\oliinyk\AppData\Local\Programs\Python\Python311\Lib\encodings\cp1252.py", line 19, in encode
    return codecs.charmap_encode(input,self.errors,encoding_table)[0]
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeEncodeError: 'charmap' codec can't encode characters in position 37-44: character maps to <undefined>
Call stack:
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "d:\Projects\nanoPhos_env\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "d:\Projects\nanoPhos_env\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "d:\Projects\nanoPhos_env\Lib\site-packages\ipykernel\ke

100%|██████████| 311/311 [00:01<00:00, 277.39it/s] 
                                                  


--- Logging error ---
Traceback (most recent call last):
  File "C:\Users\oliinyk\AppData\Local\Programs\Python\Python311\Lib\logging\__init__.py", line 1113, in emit
    stream.write(msg + self.terminator)
  File "C:\Users\oliinyk\AppData\Local\Programs\Python\Python311\Lib\encodings\cp1252.py", line 19, in encode
    return codecs.charmap_encode(input,self.errors,encoding_table)[0]
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeEncodeError: 'charmap' codec can't encode characters in position 37-46: character maps to <undefined>
Call stack:
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "d:\Projects\nanoPhos_env\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "d:\Projects\nanoPhos_env\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "d:\Projects\nanoPhos_env\Lib\site-packages\ipykernel\ke

kinases enriched: {'noreg': 294, 'upreg': 17}


# Figures 4k & 4l
Pathway enrichment (Enrichr via gseapy) of the up- (4k) and down-regulated (4l) phosphoprotein genes from the AC-vs-H differential, against the **detected-proteome background**. Six libraries (WikiPathways, GO-BP, KEGG, Reactome, MSigDB Hallmark, BioPlanet). Bars show log2 odds ratio for selected terms, colored by adjusted p. *Selected terms are carried over from v00; the full significant-term lists are printed so they can be revised on the reanalyzed data.*

In [56]:
# Figures 4k/4l (compute) - Enrichr enrichment of up/down genes vs proteome background.
from Bio import SeqIO
import gseapy as gp

def uniprot_to_gene_from_fasta(fasta_path):
    mapping = {}
    for record in SeqIO.parse(fasta_path, 'fasta'):
        header = record.description
        parts = header.split('|')
        uid = parts[1] if len(parts) >= 2 else header.split()[0]
        gn = next((f[3:] for f in header.split() if f.startswith('GN=')), None)
        if gn:
            mapping[uid] = gn
    return mapping

upreg_genes   = limma_AC_vs_H[limma_AC_vs_H['ID'] == 'up']['Gene'].unique().tolist()
downreg_genes = limma_AC_vs_H[limma_AC_vs_H['ID'] == 'down']['Gene'].unique().tolist()
print(f'up genes: {len(upreg_genes)} | down genes: {len(downreg_genes)}')

# background = all genes from the detected proteome
fasta_map = uniprot_to_gene_from_fasta(HUMAN_FASTA)
background_genes = sorted({fasta_map[u.strip()] for pg in prot['PG.ProteinGroups']
                           for u in str(pg).split(';') if u.strip() in fasta_map})
print(f'background genes: {len(background_genes)}')

gene_set_libraries = ['WikiPathway_2023_Human', 'GO_Biological_Process_2023', 'KEGG_2021_Human',
                      'Reactome_2022', 'MSigDB_Hallmark_2020', 'BioPlanet_2019']

enr_up   = gp.enrich(gene_list=upreg_genes,   gene_sets=gene_set_libraries,
                     background=background_genes, outdir=None, cutoff=0.05)
enr_down = gp.enrich(gene_list=downreg_genes, gene_sets=gene_set_libraries,
                     background=background_genes, outdir=None, cutoff=0.05)
enr_up_df   = enr_up.results[enr_up.results['Adjusted P-value'] < 0.05]
enr_down_df = enr_down.results[enr_down.results['Adjusted P-value'] < 0.05]
print(f'\nsignificant enriched terms: up={len(enr_up_df)}  down={len(enr_down_df)}')
print('\n--- UP terms (revise selection if needed) ---'); print(list(enr_up_df['Term'])[:40])
print('\n--- DOWN terms (revise selection if needed) ---'); print(list(enr_down_df['Term'])[:40])

up genes: 370 | down genes: 138
background genes: 8242

significant enriched terms: up=143  down=67

--- UP terms (revise selection if needed) ---
['Thermogenesis WP4321', 'EGF EGFR Signaling Pathway WP437', 'Common Pathways Underlying Drug Addiction WP2636', 'Regulation Of mRNA Splicing, Via Spliceosome (GO:0048024)', 'Negative Regulation Of DNA-templated Transcription (GO:0045892)', 'Negative Regulation Of Nucleic Acid-Templated Transcription (GO:1903507)', 'Regulation Of DNA-templated Transcription (GO:0006355)', 'Regulation Of Transcription By RNA Polymerase II (GO:0006357)', 'Chromatin Organization (GO:0006325)', 'Negative Regulation Of DNA Binding (GO:0043392)', 'Chromatin Remodeling (GO:0006338)', 'Cellular Response To Ionizing Radiation (GO:0071479)', 'Positive Regulation Of DNA-templated Transcription (GO:0045893)', 'Negative Regulation Of Transcription By RNA Polymerase II (GO:0000122)', 'Negative Regulation Of Gene Expression, Epigenetic (GO:0045814)', 'Regulation Of Double-

In [57]:
# Figure 4k - up-regulated pathways: cancer-relevant selection (v00 priority + keyword matches).
PURPLE_SCALE = [[0, '#5a4a6f'], [0.25, '#9970ab'], [0.5, '#c994c7'], [0.75, '#d4b9da'], [1, '#e0d0e8']]

# v00 hand-picked terms kept as priority (always shown if enriched)
UP_PRIORITY = [
    'EGF EGFR Signaling Pathway WP437', 'VEGFA VEGFR2 Signaling WP3888',
    'Signaling By ALK Fusions And Activated Point Mutants R-HSA-9725370',
    'Diseases Of Signal Transduction By Growth Factor Receptors And Second Messengers R-HSA-5663202',
    'PTEN Regulation R-HSA-6807070', 'Chromatin Remodeling (GO:0006338)']
# cancer-up / proliferative-signaling keywords (substring, case-insensitive)
UP_KEYWORDS = ['EGF', 'ERBB', 'VEGF', 'PI3K', 'AKT', 'MTOR', 'MAPK', 'ERK', 'RAS', 'RAF',
               'CELL CYCLE', 'DNA REPLICATION', 'MYC', 'E2F', 'G2M', 'G2-M', 'GROWTH FACTOR',
               'RECEPTOR TYROSINE', 'FOCAL ADHESION', 'ALK', 'PTEN', 'CHROMATIN', 'ANGIOGEN',
               'GLYCOLYSIS', 'OXIDATIVE PHOSPH', 'HYPOXIA', 'PROLIFERAT', 'INSULIN', 'WNT']

def select_cancer_terms(df, priority, keywords, n=5):
    """Pick cancer-relevant enriched terms; ONE term per keyword-theme (dedupe near-duplicates
    such as two apoptosis/cell-cycle variants), then top-n by odds ratio."""
    d = df.copy()
    d['_u'] = d['Term'].str.upper()
    cand = pd.concat([d[d['Term'].isin(priority)],
                      d[d['_u'].apply(lambda t: any(k in t for k in keywords))]]).drop_duplicates('Term')
    def theme(t):
        for k in keywords:
            if k in t:
                return k
        return t                      # priority term with no keyword -> its own theme
    cand['_theme'] = cand['_u'].apply(theme)
    cand = cand.sort_values('Odds Ratio', ascending=False).drop_duplicates('_theme').head(n)
    return cand.sort_values('Odds Ratio', ascending=True)

sel_up = select_cancer_terms(enr_up_df, UP_PRIORITY, UP_KEYWORDS, n=5)
print(f'4k: showing {len(sel_up)} up-regulated cancer-relevant terms')
print(sel_up[['Term', 'Odds Ratio', 'Adjusted P-value']].to_string(index=False))

fig = px.bar(sel_up, y='Term', x=np.log2(sel_up['Odds Ratio']), color='Adjusted P-value',
             color_continuous_scale=PURPLE_SCALE)
fig.update_layout(width=700, height=600, template='plotly_white',
                  xaxis_title='log2(Odds Ratio)', yaxis_title='')
fig.show()
#fig.write_image(r'figures/figure4/figure4k.pdf', width=700, height=600)

4k: showing 5 up-regulated cancer-relevant terms
                                                                                      Term  Odds Ratio  Adjusted P-value
                                       Regulation Of PTEN Gene Transcription R-HSA-8943724    5.976757          0.008035
                        Nuclear Events Stimulated By ALK Signaling In Cancer R-HSA-9725371    8.972603          0.018444
CREB1 Phosphorylation Thru NMDA Receptor-Mediated Activation Of RAS Signaling R-HSA-442742   10.769863          0.012474
            Regulation Of DNA Methylation-Dependent Heterochromatin Formation (GO:0090308)   21.497268          0.020278
                                                         B Cell Proliferation (GO:0042100)   32.166213          0.045134


In [58]:
# Figure 4l - down-regulated pathways: cancer-relevant, ONE term per theme (self-contained).
# Self-contained: does not depend on 4k being re-run.
PURPLE_SCALE = [[0, '#5a4a6f'], [0.25, '#9970ab'], [0.5, '#c994c7'], [0.75, '#d4b9da'], [1, '#e0d0e8']]
DOWN_PRIORITY = [
    'TGF Beta Signaling Pathway WP366', 'Regulation Of Actin Cytoskeleton WP51',
    'Regulation Of Stress Fiber Assembly (GO:0051492)', 'mRNA Splicing, Via Spliceosome (GO:0000398)',
    'Regulation Of Epithelial To Mesenchymal Transition (GO:0010717)']
DOWN_KEYWORDS = ['TGF', 'APOPTO', 'P53', 'ADHESION', 'TIGHT JUNCTION', 'ADHERENS', 'EPITHELIAL',
                 'MESENCHYMAL', 'EMT', 'ACTIN', 'CYTOSKELET', 'STRESS FIBER', 'SPLIC',
                 'CELL JUNCTION', 'HIPPO', 'CADHERIN', 'DIFFERENTIATION',
                 'EXTRACELLULAR MATRIX', 'INTERFERON', 'IMMUNE', 'COMPLEMENT']

def _select_themed(df, priority, keywords, n=5):
    d = df.copy()
    d['_u'] = d['Term'].str.upper()
    cand = pd.concat([d[d['Term'].isin(priority)],
                      d[d['_u'].apply(lambda t: any(k in t for k in keywords))]]).drop_duplicates('Term')
    def theme(t):
        for k in keywords:
            if k in t:
                return k
        return t
    cand['_theme'] = cand['_u'].apply(theme)
    cand = cand.sort_values('Odds Ratio', ascending=False).drop_duplicates('_theme').head(n)
    return cand.sort_values('Odds Ratio', ascending=True)

sel_down = _select_themed(enr_down_df, DOWN_PRIORITY, DOWN_KEYWORDS, n=5)
print(f'4l: showing {len(sel_down)} down-regulated cancer-relevant terms (one per theme)')
print(sel_down[['Term', '_theme', 'Odds Ratio', 'Adjusted P-value']].to_string(index=False))

fig = px.bar(sel_down, y='Term', x=np.log2(sel_down['Odds Ratio']), color='Adjusted P-value',
             color_continuous_scale=PURPLE_SCALE)
fig.update_layout(width=700, height=600, template='plotly_white',
                  xaxis_title='log2(Odds Ratio)', yaxis_title='')
fig.show()
fig.write_image(r'figures/figure4/figure4l.pdf', width=1200, height=600)

4l: showing 5 down-regulated cancer-relevant terms (one per theme)
                                                                                                Term               _theme  Odds Ratio  Adjusted P-value
                                                                     Cell to cell adhesion signaling             ADHESION   14.985185          0.040100
                                                            MicroRNAs in muscle cell differentiation      DIFFERENTIATION   17.986667          0.035136
MFAP5 Effect On Permeability And Motility Of Endothelial Cells Via Cytoskeleton Rearrangement WP4560           CYTOSKELET   24.161194          0.015868
                                           Apoptotic Cleavage Of Cell Adhesion Proteins R-HSA-351906               APOPTO   29.992593          0.017163
                                   Positive Regulation Of Extracellular Matrix Assembly (GO:1901203) EXTRACELLULAR MATRIX   35.995556          0.045642


In [59]:
# === PRIDE MetaInfo export (run after all panels above) ===
import sys; sys.path.insert(0, r"src")
from metainfo_export import dump_panel
FIG = 4   # figure number (single source of truth for sheet labels)
from core import count_sites_per_sample_ptm_report, process_ptm_site_report
import numpy as np, pandas as pd
_META={"Protein_group","Gene_group","PTM_0_aa","PTM_pos","PTM_mult123","PTM_flank","PTM_Collapse_key","PTM_localization","UPD_seq"}
def _try(fn, sheet):
    try: fn()
    except Exception as e: print(f"  [SKIP {sheet}] {type(e).__name__}: {e}")
def _depth(dct, key):
    _r=[]
    for k in sorted(dct):
        for rep,(samp,c) in enumerate(count_sites_per_sample_ptm_report(dct[k]).items(),1):
            _r.append({"Raw file":samp, key:k, "Replicate":rep, "Number of class I sites":int(c)})
    return pd.DataFrame(_r)

def _4a():
    a=_depth(ff,"Condition_ng"); a["Material"]="FF"
    b=_depth(ffpe,"Condition_ng"); b["Material"]="FFPE"
    dump_panel(pd.concat([a,b],ignore_index=True), f"Figure {FIG}a")
_try(_4a,f"Figure {FIG}a")
def _4b():
    cols={}
    for lab,dct in [("ff",ff),("ffpe",ffpe)]:
        for ng in sorted(dct):
            sd=process_ptm_site_report(dct[ng],cutoff=0.75)["site_data"]; sc=[c for c in sd.columns if c not in _META]
            lin=np.power(2.0,sd[sc]); cv=(lin.std(axis=1)/lin.mean(axis=1)).where(lin.notna().sum(axis=1)>=len(sc)).dropna()
            cols[f"{lab}_{ng}ng"]=cv.reset_index(drop=True)
    dump_panel(pd.DataFrame(cols), f"Figure {FIG}b")
_try(_4b,f"Figure {FIG}b")
_try(lambda: dump_panel(cmp[["FFPE_mean","FF_mean"]], f"Figure {FIG}c, {FIG}f"), f"Figure {FIG}c, {FIG}f")
_try(lambda: dump_panel(pca_df.rename(columns={"x":"PC1","y":"PC2"}), f"Figure {FIG}d"), f"Figure {FIG}d")
_try(lambda: dump_panel(res.rename(columns={"pathway":"ID"})[["ID","n","Pearson","Spearman","highlight"]], f"Figure {FIG}e"), f"Figure {FIG}e")
_try(lambda: dump_panel(_pdf.rename(columns={"x":"PC1","y":"PC2"}), f"Figure {FIG}h"), f"Figure {FIG}h")
_try(lambda: dump_panel(limma_AC_vs_H, f"Figure {FIG}i"), f"Figure {FIG}i")
_try(lambda: dump_panel(kinase_df.rename_axis("kinase").reset_index(), f"Figure {FIG}j"), f"Figure {FIG}j")
_try(lambda: dump_panel(sel_up, f"Figure {FIG}k"), f"Figure {FIG}k")
_try(lambda: dump_panel(sel_down, f"Figure {FIG}l"), f"Figure {FIG}l")
print(f"Figure {FIG} export done.")


  [MetaInfo] wrote 'Figure 4a'  (42 rows x 5 cols)
Dropped 201 non-Phospho rows. Mods in input: ['Carbamidomethyl (C)', 'Oxidation (M)', 'Phospho (STY)']. Mods kept: ['Phospho (STY)']
Per-run mask (cutoff=0.75): 0 of 1,402 intensity cells masked (0.0%).
Deduplicated multi-protein rows: 666 → 662.
Final: 662 sites × 3 samples.
Dropped 354 non-Phospho rows. Mods in input: ['Carbamidomethyl (C)', 'Oxidation (M)', 'Phospho (STY)']. Mods kept: ['Phospho (STY)']
Per-run mask (cutoff=0.75): 0 of 2,865 intensity cells masked (0.0%).
Deduplicated multi-protein rows: 1,435 → 1,431.
Final: 1,431 sites × 3 samples.
Dropped 446 non-Phospho rows. Mods in input: ['Carbamidomethyl (C)', 'Oxidation (M)', 'Phospho (STY)']. Mods kept: ['Phospho (STY)']
Per-run mask (cutoff=0.75): 0 of 5,211 intensity cells masked (0.0%).
Deduplicated multi-protein rows: 2,634 → 2,611.
Final: 2,611 sites × 3 samples.
Dropped 1,005 non-Phospho rows. Mods in input: ['Carbamidomethyl (C)', 'Oxidation (M)', 'Phospho (STY)']. 